# rel-f1 — GPU side

This notebook is only a runner: clone the repo, install dependencies, call the scripts in
`src/`. No logic lives here, otherwise the project would have two codebases.

**Settings -> Accelerator -> GPU T4 x2, and Internet -> On. Then Run All.**

Every training cell skips work whose results are already in `output/evaluation.csv`, so
running the notebook twice does not duplicate rows or waste GPU time.


## 1. Repo

In [ ]:
!rm -rf /kaggle/working/IIS-RDL-Project
%cd /kaggle/working
!git clone --depth 1 https://github.com/temelkovaelena/IIS-RDL-Project.git
%cd /kaggle/working/IIS-RDL-Project
!git log --oneline -1


## 2. Dependencies

Kaggle already ships torch. On top of it go the PyG stack and a compiled sampling backend:
`NeighborLoader` cannot sample without either `pyg-lib` or `torch-sparse`, and which wheel
fits depends on the exact torch and CUDA build, so the version is derived rather than
hard-coded.


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.is_available())


In [ ]:
!pip install -q relbench torch_geometric pytorch_frame sentence-transformers


In [ ]:
import importlib
import subprocess
import sys

import torch

TORCH = torch.__version__.split("+")[0]
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"


def sampling_backend():
    importlib.invalidate_caches()
    for name in ("pyg_lib", "torch_sparse"):
        try:
            importlib.import_module(name)
            return name
        except ImportError:
            continue
    return None


# Wheels are not published for every torch release. Try the running version first, then
# fall back to recent ones, whose builds are usually compatible.
candidates = [TORCH] + [v for v in ("2.7.0", "2.6.0", "2.5.0", "2.4.0") if v != TORCH]
for version in candidates:
    if sampling_backend():
        break
    index = f"https://data.pyg.org/whl/torch-{version}+{CUDA}.html"
    print(f"trying {index}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyg-lib", "torch-sparse", "-f", index]
    )

print("sampling backend:", sampling_backend() or "NONE")


## 3. Environment check

Without a sampling backend the loader fails on the first batch, so it is better to stop here
with a clear message than deep inside a traceback later.


In [ ]:
import relbench
import torch_frame
import torch_geometric
from relbench.modeling.graph import make_pkey_fkey_graph
from relbench.modeling.utils import get_stype_proposal

print("relbench", relbench.__version__)
print("torch_frame", torch_frame.__version__)
print("torch_geometric", torch_geometric.__version__)

backend = sampling_backend()
print("sampling backend:", backend)
assert backend is not None, (
    "No sampling backend. NeighborLoader needs pyg-lib or torch-sparse. "
    "Re-run the install cell above and check its output."
)


## 4. Build the graph

Row -> node, table -> node type, foreign key -> typed edge, timestamp -> node time. Text
columns (driver, constructor and circuit names) need an embedder supplied up front, or
torch_frame refuses to build the dataset.


In [ ]:
import sys

sys.path.insert(0, "src")
from features import text_embedder_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = relbench.load_dataset("rel-f1")
db = dataset.get_db()
col_to_stype = get_stype_proposal(db)

data, col_stats = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype,
    text_embedder_cfg=text_embedder_config(device=device),
    cache_dir="/kaggle/working/cache",
)

print(data)
print()
print("node types:", sorted(data.node_types))
print("edge types:", len(data.edge_types))


In [ ]:
# The edge type names must match the ones measured locally, otherwise the statistics in
# data/graph_stats.json describe a different graph from the one being trained on.
import json

stats = json.load(open("data/graph_stats.json"))
local = set(stats["edge_types"])
gpu = {f"{s} -{r}-> {d}" for s, r, d in data.edge_types}

print("edge types match:", local == gpu)
print("nodes:", stats["n_nodes"], "| edges:", stats["n_edges_undirected"])


## 5. How training is run

`run()` calls `src/train.py` as a subprocess and shows its output. `done()` checks whether a
result is already recorded, so nothing is trained twice.

Watch for `temporal check on first batch` in the output: it prints before any training
happens, so a leak in the sampler shows up immediately.


In [ ]:
import os
import shutil

import pandas as pd

# Results live OUTSIDE the cloned repo. Cell 2 deletes the clone on every run, so anything
# written inside it would be lost the next time the notebook starts.
WORK = "/kaggle/working/results"
os.makedirs(WORK, exist_ok=True)
RESULTS = f"{WORK}/evaluation.csv"
SWEEP = f"{WORK}/sweep.csv"

# Seed the working copy from the repo once, so the committed seeds 0-2 count as done.
if not os.path.exists(RESULTS):
    shutil.copy("output/evaluation.csv", RESULTS)
    print(f"seeded {RESULTS} from the repo")

CACHE = ["--cache-dir", "/kaggle/working/cache"]


def run(args):
    r = subprocess.run([sys.executable, "src/train.py", *args], capture_output=True, text=True)
    print(r.stdout[-1200:])
    if r.returncode != 0:
        print("FAILED\n", r.stderr[-2500:])
    return r.returncode == 0


def done(task, arm, seed, path=RESULTS, split="test"):
    """True if this configuration is already recorded, so Run All stays idempotent.

    The sweep writes validation rows only, so it passes split="val".
    """
    if not os.path.exists(path):
        return False
    d = pd.read_csv(path)
    return bool(
        ((d.task == task) & (d.arm == arm) & (d.seed == seed) & (d.split == split)).any()
    )


print("already recorded:")
print(pd.read_csv(RESULTS).groupby(["task", "arm"])["seed"].nunique().to_string())


## 6. Seeds

The repo holds seeds 0-2 for each task. Two more take every configuration to five, which is
what decides whether the small gap on `driver-dnf` is real or just seed-to-seed noise.


In [ ]:
for task in ["driver-top3", "driver-dnf", "driver-position"]:
    for seed in [0, 1, 2, 3, 4]:
        if done(task, "gnn", seed):
            print(f"skip {task} seed {seed}")
            continue
        print(f"\n===== {task}  seed {seed} =====")
        run(["--task", task, "--seed", str(seed), "--epochs", "20",
             "--out", RESULTS, *CACHE])


## 7. Hyperparameter search

Both tabular models stop early on validation, while the graph model has so far run with
fixed settings. That makes the comparison unfair to the graph, so this searches a small grid.

Scored on **validation only**, so the test split stays untouched during model selection, and
written to a separate file so `evaluation.csv` keeps only reportable numbers.


In [ ]:
import itertools

SWEEP = "output/sweep.csv"

for lr, channels in itertools.product([0.01, 0.005, 0.001], [64, 128]):
    arm = f"gnn-lr{lr}-ch{channels}"
    if done("driver-top3", arm, 0, path=SWEEP, split="val"):
        print(f"skip {arm}")
        continue
    print(f"\n===== lr={lr}  channels={channels} =====")
    ok = run([
        "--task", "driver-top3", "--seed", "0", "--epochs", "20",
        "--lr", str(lr), "--channels", str(channels),
        "--splits", "val", "--sweep", "--arm", arm,
        "--out", SWEEP, *CACHE,
    ])
    if not ok:
        break


In [ ]:
if not os.path.exists(SWEEP):
    print("no sweep.csv - check the output of the cell above")
else:
    s = pd.read_csv(SWEEP).sort_values("value", ascending=False)
    print(s[["lr", "channels", "value"]].to_string(index=False))
    top = s.iloc[0]
    print()
    print(f"best on val: lr={top.lr}, channels={top.channels}, roc_auc={top.value:.4f}")
    print("current default is lr=0.005, channels=128")


## 8. Hop depth

One layer means the model sees only the driver's own rows. Two is where the constructor and
the race first become reachable, three where the circuit does.

All three depths run at **batch size 128**, not the default 512. At depth 3 the sampled
subgraph is large enough to exhaust a 16 GB GPU, and batch size is the right thing to lower:
it changes how many seeds are processed per step, not what any single seed sees. Lowering
the fanout instead would change the neighbourhood itself and confound the comparison.

Depth 2 is repeated here rather than reused from section 6, so all three share identical
settings.


In [ ]:
# The allocator fragments badly on the large depth-3 batches; this helps it reuse blocks.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

ABLATION_BATCH = "128"

for layers in [1, 2, 3]:
    arm = f"gnn-d{layers}"
    for seed in [0, 1, 2]:
        if done("driver-top3", arm, seed):
            print(f"skip {arm} seed {seed}")
            continue
        print(f"\n===== depth {layers}  seed {seed} =====")
        run([
            "--task", "driver-top3", "--seed", str(seed), "--epochs", "20",
            "--num-layers", str(layers), "--batch-size", ABLATION_BATCH,
            "--arm", arm, "--out", RESULTS, *CACHE,
        ])


## 9. Results

Everything together, and copies in `/kaggle/working` for the Output panel on the right.
Download both files before the session ends.


In [ ]:
d = pd.read_csv(RESULTS)
t = d[d.split == "test"]

print("main comparison, test:")
main = t[t.arm.isin(["lgbm-flat", "lgbm-engineered", "gnn"])]
print(main.pivot_table(index=["task", "metric"], columns="arm", values="value",
                      aggfunc=["mean", "std"]).round(4).to_string())

print()
print("hop depth, driver-top3 test (batch 128, matched settings):")
abl = t[(t.task == "driver-top3") & (t.arm.str.match(r"gnn-d\d"))]
print(abl.groupby("arm")["value"].agg(["mean", "std", "count"]).round(4).to_string()
      if len(abl) else "  no ablation rows yet")

print()
print("seeds per configuration:")
print(t.groupby(["task", "arm"])["seed"].nunique().to_string())

# Copy to /kaggle/working itself, which is what the Output panel lists.
shutil.copy(RESULTS, "/kaggle/working/evaluation.csv")
if os.path.exists(SWEEP):
    shutil.copy(SWEEP, "/kaggle/working/sweep.csv")
    print("\nsaved: evaluation.csv and sweep.csv")
else:
    print("\nsaved: evaluation.csv (no sweep rows)")
print("Download both from the Output panel before the session ends.")
